In [29]:
import json
import os
import shutil
from pathlib import Path

import cv2
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import torchvision.models.video as video_models

In [ ]:
datasetpath = Path("/kaggle/input/datasets/sarbagyashakya/basketball-51-dataset/Basketball_51 dataset")

modelpath = Path("/kaggle/input/models/spicybigmac/basketballmodel/pytorch/default/5/gamestrat.pt")
manifestpath = Path("/kaggle/input/datasets/spicybigmac/splitmanifest/split_manifest.json")

frames = 16
workers = 4
batchsize = 1
savepath = Path("/kaggle/working/explanations")
savepath.mkdir(parents=True, exist_ok=True)
camcache = savepath / "gradcam_cache"
gifpath = savepath / "gradcam_examples"
camcache.mkdir(parents=True, exist_ok=True)
gifpath.mkdir(parents=True, exist_ok=True)

alpha = 0.5
colormap = cv2.COLORMAP_JET
fps = 8
examples_per_category = 5   # 2p, 3p, free throw, and mid-range
perturbbatchsize = 8
kernelsize = 7
perturbationstep = 0.10
sidelen = 60
seed = 42

np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True

print(f"model: {modelpath}")
print(f"split manifest: {manifestpath}")

In [ ]:
evaltrans = v2.Compose([
    v2.Resize([112, 112], antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.43216, 0.394666, 0.37645], std=[0.22803, 0.22145, 0.216989])
])

with open(manifestpath) as f:
    split_manifest = json.load(f)

class_names = sorted([
    directory.name for directory in datasetpath.iterdir()
    if directory.is_dir() and not directory.name.startswith(".")
])
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

testpaths = [datasetpath / relative for relative in split_manifest["test"]["clips"]]
testlabels = [class_to_idx[path.parent.name] for path in testpaths]
missing = [path for path in testpaths if not path.exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} manifest clips are missing; first missing path: {missing[0]}")

class BasketballDataset(Dataset):
    def __init__(self, videopaths, labels, classes, frames=16):
        self.videopaths = list(videopaths)
        self.labels = list(labels)
        self.classes = list(classes)
        self.goalframes = frames

    def __len__(self):
        return len(self.videopaths)

    def __getitem__(self, idx):
        frames = self._load_video(self.videopaths[idx])
        video = torch.from_numpy(frames).permute(0, 3, 1, 2)
        video = evaltrans(video).permute(1, 0, 2, 3)
        return video, self.labels[idx]

    def _load_video(self, path):
        cap = cv2.VideoCapture(str(path))
        framecount = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if framecount <= 0:
            cap.release()
            raise RuntimeError(f"Could not read frames from {path}")

        if framecount < self.goalframes:
            frame_indices = np.pad(
                np.arange(framecount),
                (0, self.goalframes - framecount),
                mode="edge"
            )
        else:
            frame_indices = np.linspace(0, framecount - 1, self.goalframes, dtype=int)

        requested = list(frame_indices)
        unique_requested = set(requested)
        decoded = {}
        frame_idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_idx in unique_requested:
                decoded[frame_idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_idx += 1
        cap.release()

        if not decoded:
            raise RuntimeError(f"No requested frames could be decoded from {path}")
        last = decoded[min(decoded)]
        output = []
        for frame_idx in requested:
            if frame_idx in decoded:
                last = decoded[frame_idx]
            output.append(last)
        return np.stack(output)

print(f"using {len(testpaths)} clips from {len(split_manifest['test']['games'])} games")

In [ ]:
testdata = BasketballDataset(testpaths, testlabels, class_names, frames)
testloader = DataLoader(
    testdata,
    batch_size=batchsize,
    shuffle=False,
    num_workers=workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(workers > 0)
)
classes = len(class_names)
print(f"final test set: {len(testdata)} clips\nclass counts: {np.bincount(testlabels, minlength=classes).tolist()}")

In [ ]:
model = video_models.r2plus1d_18(weights=None)
infeatures = model.fc.in_features
model.fc = nn.Linear(infeatures, classes)

try:
    checkpoint = torch.load(modelpath, map_location="cpu", weights_only=True)
except TypeError:
    checkpoint = torch.load(modelpath, map_location="cpu")

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    checkpoint = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    checkpoint = checkpoint["state_dict"]

state_dict = {
    (key[7:] if key.startswith("module.") else key): value
    for key, value in checkpoint.items()
}

model.load_state_dict(state_dict, strict=True)

model.eval()
print("loaded checkpoint successfully")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print(f"model loaded on {device}")

baseline_correct = 0
baseline_total = 0
with torch.no_grad():
    for videos, labels in testloader:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        predictions = model(videos).argmax(dim=1)
        baseline_correct += (predictions == labels).sum().item()
        baseline_total += labels.size(0)

baseline_accuracy = baseline_correct / baseline_total
print(f"final test accuracy: {baseline_accuracy:.3%} ({baseline_correct}/{baseline_total})")

model loaded on cuda
Final test accuracy: 69.313% (777/1121)


In [ ]:
# explanations

class GradCam:
    def __init__(self, model, layer):
        self.model = model
        self.layer = layer
        self.grads, self.activations = None, None

        self.forw = layer.register_forward_hook(self.save_activations)
        self.back = layer.register_full_backward_hook(self.save_grads)
        
    def save_activations(self, module, input, output):
        self.activations = output.detach()
    def save_grads(self, module, gradinput, gradoutput):
        self.grads = gradoutput[0].detach()
    
    def remove_hooks(self):
        self.forw.remove()
        self.back.remove()

    def __call__(self, inp, targetclass=None):
        self.model.eval()
        self.model.zero_grad()

        output = self.model(inp)

        if targetclass is None:
            targetclass = output.argmax(dim=1).item()

        score = output[0,targetclass]
        score.backward()

        activations = self.activations[0]
        grads = self.grads[0]
        
        weights = grads.mean(dim=[1,2,3], keepdim=True)

        cam = (weights*activations).sum(dim=0)
        cam = nn.functional.relu(cam)

        cam -= cam.min()
        cam /= (cam.max() + 1e-8)

        t,h,w = inp.shape[2:]
        cam = nn.functional.interpolate(cam.unsqueeze(0).unsqueeze(0), size=[t,h,w], mode="trilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()

        prob = torch.softmax(output, dim=1)[0, targetclass].item()
        return cam, targetclass, prob

In [ ]:
# helpers

def denormalize(vidtensor, mean, std):
    mean = torch.tensor(mean).view(-1, 1, 1, 1)
    std = torch.tensor(std).view(-1, 1, 1, 1)
    
    vid = vidtensor.cpu() * std + mean
    vid = torch.clamp(vid, 0, 1)
    return vid.numpy()


def frame_to_uint8(frame):
    return (np.clip(frame, 0, 1) * 255).astype(np.uint8)

# generate/save image

def overlay(vid, cam):
    C, T, H, W = vid.shape
    frames = []
    
    for i in range(T):
        frame = np.transpose(vid[:,i,:,:], (1, 2, 0)) # transposes [c,h,w] to [h,w,c]
        if frame.shape[2] == 1:
            # if its black and white 
            frame = np.repeat(frame, 3, axis=2)
        
        # cam for this frame
        camframe = cam[i]
        
        # cam -> heatmap
        heatmap = cv2.applyColorMap(np.uint8(255 * camframe), colormap)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
        
        # overlap heatmap onto frame
        overlay = frame*(1-alpha) + heatmap*alpha
        overlay = np.clip(overlay, 0, 1)
        
        frames.append(frame_to_uint8(overlay))
    
    return frames


def save_gif(frames, path):
    imageio.mimsave(path, frames, fps=fps)
    print(f"saved gif at: {path}")


def broad_category(class_name):
    if class_name.startswith("2p"):
        return "2p"
    if class_name.startswith("3p"):
        return "3p"
    if class_name.startswith("ft"):
        return "ft"
    if class_name.startswith("mp"):
        return "mid_range"
    raise ValueError(f"Unknown Basketball-51 class: {class_name}")

# main function

def visualize_gradcam(model, testloader, targetlayer, mean=(0.43216, 0.394666, 0.37645), std=(0.22803, 0.22145, 0.216989)):
    os.makedirs(savepath, exist_ok=True)
    
    # init gradcam
    gradcam = GradCam(model, targetlayer)
    
    out = []
    saved_by_category = {"2p": 0, "3p": 0, "ft": 0, "mid_range": 0}
    
    print(f"grad camming in {savepath}")
    
    clip_index = 0
    for videos, labels in testloader:
        videos = videos.to(device)
        labels = labels.cpu().numpy()
        
        for i in range(videos.shape[0]):
            video = videos[i:i+1]  # keep batch dim for model
            
            # run the gradcam class and prepare frames
            cam, predictedclass, prob = gradcam(video)
            trueclass = int(labels[i])
            
            true_name = testdata.classes[trueclass]
            predicted_name = testdata.classes[predictedclass]
            category = broad_category(true_name)

            campath = camcache / f"test_{clip_index:05d}.npy"
            np.save(campath, cam.astype(np.float16))

            if saved_by_category[category] < examples_per_category:
                vid = denormalize(video[0], mean, std)
                overlaid = overlay(vid, cam)
                example_number = saved_by_category[category] + 1
                example_path = gifpath / (
                    f"{category}_{example_number:02d}_test{clip_index:05d}_"
                    f"true-{true_name}_pred-{predicted_name}_p{prob:.3f}.gif"
                )
                save_gif(overlaid, example_path)
                saved_by_category[category] += 1

            out.append({
                "cam_path": str(campath),
                "predicted_class": predictedclass,
                "true_class": trueclass,
                "correct": predictedclass == trueclass,
                "probability": prob,
                "test_index": clip_index,
                "relative_path": split_manifest["test"]["clips"][clip_index],
            })
            if (clip_index + 1) % 25 == 0 or clip_index + 1 == len(testdata):
                print(f"Grad-CAM: {clip_index + 1}/{len(testdata)} test clips")
            clip_index += 1
    
    gradcam.remove_hooks()
    if clip_index != len(testdata):
        raise RuntimeError(f"generated {clip_index} maps for {len(testdata)} test clips")
    if any(value < examples_per_category for value in saved_by_category.values()):
        raise RuntimeError(f"could not save five examples per category: {saved_by_category}")

    with open(savepath / "gradcam_metadata.json", "w") as file:
        json.dump(out, file, indent=2)
    archive = shutil.make_archive(
        str(savepath / "gradcam_examples"), "zip", root_dir=gifpath
    )
    print(f"completed all {len(out)} test clips")
    print(f"downloadable example archive: {archive}")

    return out

In [ ]:
out = visualize_gradcam(model, testloader, model.layer4)

In [ ]:
# faithfulness verification

def perturb(vid, mask, kernelsize):
    C, T, H, W = vid.shape
    perturbed = vid.copy()

    for i in range(T):
        frame = vid[:,i,:,:]
        blurred = np.zeros_like(frame)
        for j in range(C):
            blurred[j] = cv2.GaussianBlur(frame[j].astype(np.float32), (kernelsize,kernelsize), 0)

        m = mask[i:i+1,:,:]
        perturbed[:,i,:,:] = frame*(1-m) + blurred*m

    return perturbed

def faithfulness(model, examples, kernelsize):
    # store predicted-class confidence after salient and random perturbation.
    salient, random = [], [] 

    for example in examples:
        test_index = example["test_index"]
        vid, _ = testdata[test_index]
        vid = vid.numpy()
        cam = np.load(example["cam_path"]).astype(np.float32)
        predclass = example["predicted_class"]
        prob = example["probability"]
        
        flatcam = cam.flatten()
        pixels = flatcam.shape[0]  # one importance value per spatiotemporal location

        salientidxs = np.argsort(flatcam)[::-1]
        randomidxs = np.random.default_rng(seed + test_index).permutation(pixels)
        perturbed_variants = []

        for i in range(1,11):
            toblur = int(pixels*i*perturbationstep)
            
            salientmask = np.zeros([pixels])
            salientmask[salientidxs[:toblur]] = 1
            salientmask = salientmask.reshape(cam.shape)

            randommask = np.zeros([pixels])
            randommask[randomidxs[:toblur]] = 1
            randommask = randommask.reshape(cam.shape)

            perturbed_variants.append(perturb(vid, salientmask, kernelsize))
            perturbed_variants.append(perturb(vid, randommask, kernelsize))

        perturbed_probabilities = []
        for start in range(0, len(perturbed_variants), perturbbatchsize):
            batch = torch.from_numpy(
                np.stack(perturbed_variants[start:start + perturbbatchsize])
            ).to(device)
            with torch.inference_mode():
                probabilities = torch.softmax(model(batch), dim=1)[:, predclass]
            perturbed_probabilities.extend(probabilities.cpu().tolist())

        salient.append([prob, *perturbed_probabilities[0::2]])
        random.append([prob, *perturbed_probabilities[1::2]])
        if (test_index + 1) % 25 == 0 or test_index + 1 == len(testdata):
            print(f"Faithfulness: {test_index + 1}/{len(testdata)} test clips")

    return salient, random


In [ ]:
# get faithfulness

salient, random = faithfulness(model, out, kernelsize)

salientavg = np.mean(salient, axis=0)
randomavg = np.mean(random, axis=0)

In [ ]:
print("salient mean confidence:", salientavg)
print("random mean confidence:", randomavg)

In [ ]:
# plot faithfulness

xvals = np.arange(11) * perturbationstep * 100
xfrac = xvals / 100

plt.plot(xvals, salientavg, label="salient", color="blue")
plt.plot(xvals, randomavg, label="random", color="red", ls="--")

plt.xlabel("% of Pixels Perturbed")
plt.ylabel("Confidence of Original Prediction")
plt.title("Confidence of Original Prediction After Perturbations")

plt.legend()
plt.tight_layout()
plt.savefig(savepath / "confidencegraph.png", dpi=300)
plt.show()

diffs = np.abs(salientavg-randomavg)
plt.plot(xvals, diffs)

plt.xlabel("% of Pixels Perturbed")
plt.ylabel("Difference Between Salient and Average Confidence of Original Prediction")
plt.title("Difference Between Confidences of Original Prediction")

plt.tight_layout()
plt.savefig(savepath / "differencegraph.png", dpi=300)
plt.show()

In [ ]:
# verify that the middle section of the screen actually contributes to accuracy

def blur(vid, mask, kernelsize):
    B, C, T, H, W = vid.shape # B will always be 1
    blurred = vid.copy()

    for i in range(T):
        frame = vid[:,:,i,:,:]
        blurredframe = np.zeros_like(frame)
        for j in range(C):
            blurredframe[0,j] = cv2.GaussianBlur(frame[0,j].astype(np.float32), (kernelsize,kernelsize), 0)

        blurred[:,:,i,:,:] = frame*(1-mask) + blurredframe*mask

    return blurred


def verify(model, testloader, kernelsize, sidelen):
    originalcorrect, midcorrect, randcorrect, processed = 0, 0, 0, 0

    for videos, labels in testloader:
        videos = videos.to(device)
        labels = labels.to(device)
        
        for i in range(videos.shape[0]):
            video = videos[i:i+1].cpu().numpy()

            B, C, T, H, W = video.shape
            
            midmask, randmask = np.zeros_like(video[:,:,0,:,:]), np.zeros_like(video[:,:,0,:,:])
            midmask[:,:,int((H-sidelen)/2):int((H+sidelen)/2),int((W-sidelen)/2):int((W+sidelen)/2)] = 1

            top, left = np.random.randint(0, H-sidelen+1), np.random.randint(0, W-sidelen+1)
            randmask[:,:,top:top+sidelen,left:left+sidelen] = 1
            
            midtensor = torch.from_numpy(blur(video, midmask, kernelsize)).to(device)
            randtensor = torch.from_numpy(blur(video, randmask, kernelsize)).to(device)

            with torch.no_grad():
                predoriginal = model(videos[i:i+1]).argmax(dim=1)
                predmid = model(midtensor).argmax(dim=1)
                predrand = model(randtensor).argmax(dim=1)

                originalcorrect += (predoriginal == labels[i:i+1]).sum().item()
                midcorrect += (predmid == labels[i:i+1]).sum().item()
                randcorrect += (predrand == labels[i:i+1]).sum().item()

            processed += 1

    return (
        originalcorrect / processed,
        midcorrect / processed,
        randcorrect / processed,
        processed,
    )

In [ ]:
# compare the fixed 60x60 centre blur with a random blur.
originalacc, midacc, randacc, blur_n = verify(model, testloader, kernelsize, sidelen)

In [ ]:
print(f"original={originalacc:.3%}, centre blur={midacc:.3%}, random blur={randacc:.3%}, n={blur_n}")

In [ ]:
plt.bar(["Original", "Centre Blur", "Random Blur"], [originalacc, midacc, randacc])
plt.ylabel("Classification Accuracy")
plt.title("Effect of Centre and Random Blurring")
plt.tight_layout()
plt.savefig(savepath / "central_vs_random.png", dpi=300)
plt.show()